# 第 15 章：多因子回归与历史归因

本 Notebook 是章节配套实证草稿，使用冻结的 French NoDur、FF3 与 FF5 月度文件。它不是策略回测、收益预测或第 15 章完整性验收。

资产收益和因子数据按十进制月度简单收益读取。样本固定为 2010 年 1 月至 2019 年 12 月，共 120 个连续月度 PeriodIndex 区间。资产总收益减去同月 RF 一次；已发布的 Mkt-RF、SMB、HML、RMW、CMA 原样进入回归。

**I2 默认离线路线：CONSTRUCTED 教学计算。** French 数据未随包提供，权利/用途 unknown；真实市场分支显式跳过，不能据此声称完整实证复现。原文历史数字只是章节背景，非本次输出。A10-A design accepted / execution blocked；A10-A.1 protocol technically accepted pending permissions and execution authorization；A10-B 未授权。静态图如显示则为已冻结教学参考，不表示本次重新生成。


In [ ]:
from pathlib import Path
import sys, os, uuid
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start inside the reader package tree")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch15_factor_models_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
os.chdir(ROOT)
# French branches remain disabled until data rights and execution authorization are resolved.
RUN_FRENCH = False
print("CONSTRUCTED teaching route; French market branches SKIPPED; outputs:", RUN_DIR)

from pathlib import Path
import sys

import pandas as pd


from quant_course.factor_models import fit_monthly_factor_model
from ch15_factor_models import (
    build_summary,
    fit_models,
    load_sample,
    make_figures,
    write_artifacts,
)
from IPython.display import display


## 1. 冻结样本与来源核验

工作流读取本地 manifest 指定的处理文件，并调用共享 SHA-256 校验器核对字节。它不下载或改写源数据，也不把缺月压缩成相邻观察行。NoDur 是 French 十行业价值加权组合中的非耐用品行业总收益。

In [ ]:
import numpy as np
rng = np.random.default_rng(20261015)
idx = pd.period_range("2010-01", periods=120, freq="M")
factors = pd.DataFrame(rng.normal(0, .025, (120, 5)), index=idx, columns=["Mkt-RF", "SMB", "HML", "RMW", "CMA"])
factors["RF"] = .001
frames = {"ff3": factors[["Mkt-RF", "SMB", "HML", "RF"]].copy(), "ff5": factors.copy()}
total = pd.Series(.001 + .002 + factors["Mkt-RF"] + .3*factors["HML"] + rng.normal(0, .01, 120), index=idx, name="CONSTRUCTED asset")
print("CONSTRUCTED seeded factor-regression inputs; French branch skipped")
assert len(total) == 120 and frames["ff5"].index.equals(total.index)


收益率与因子是十进制月度数值，例如 0.01 表示月度 1%。PeriodIndex 标记月份区间，不表示精确的公告、观测顺序或执行时间。FF3 与 FF5 使用各自发布的因子快照，因此相同名称的市场和 SMB 序列也不预设完全相同。

In [ ]:
display(total.head().to_frame())
display(frames["ff3"].head())
display(frames["ff5"].head())

## 2. 普通最小二乘与因子暴露

Ordinary Least Squares (OLS，普通最小二乘) 在每个模型中估计月度超额收益对因子的线性关系。CAPM 使用市场超额收益；FF3 使用市场、Small Minus Big (SMB，小盘减大盘) 和 High Minus Low (HML，高账面市值比减低账面市值比)；FF5 再加入 Robust Minus Weak (RMW，盈利稳健减盈利弱) 和 Conservative Minus Aggressive (CMA，投资保守减投资积极)。

下一个单元先直接调用共享回归核心拟合 CAPM，再由章节工作流拟合三种模型。两条路径应给出相同 CAPM 系数；Notebook 不复制估计器实现。

In [ ]:
core_capm = fit_monthly_factor_model(
    total,
    frames["ff3"],
    factor_columns=("Mkt-RF",),
    hac_maxlags=6,
)
fits = fit_models(total, frames, hac_maxlags=6)

assert core_capm.coefficient_table.equals(fits["capm"].coefficient_table)
print("共享核心和章节工作流的 CAPM 系数表一致。")
print("模型：", list(fits))

## 3. 月度截距与不确定性

Heteroskedasticity and Autocorrelation Consistent (HAC，异方差与自相关稳健) 标准误采用 Bartlett 核，带宽按月度观察行计。下表将 HAC lag 设为 3、6、12 时的截距点估计、标准误和 95% 区间并列。

区间与 p 值使用 t 参考分布和有限样本修正，是依赖条件的近似计算；软件输出本身不能验证这些条件，也不能证明经济上可实施的 alpha。

In [ ]:
summary = build_summary(total, frames, fits, constructed=True)

alpha_rows = []
for name in ("capm", "ff3", "ff5"):
    model = summary["models"][name]
    for lag, values in model["alpha_hac_lag_sensitivity"].items():
        alpha_rows.append({
            "model": name,
            "HAC lags (months)": int(lag),
            "alpha (% per month)": values["estimate"] * 100,
            "HAC standard error (% per month)": values["standard_error_hac"] * 100,
            "95% CI low (% per month)": values["ci_low_hac"] * 100,
            "95% CI high (% per month)": values["ci_high_hac"] * 100,
        })

alpha_sensitivity = pd.DataFrame(alpha_rows)
display(alpha_sensitivity.pivot(
    index="model",
    columns="HAC lags (months)",
    values="alpha (% per month)",
))

CONSTRUCTED 输出由上方固定种子计算；原章节历史截距数字不属于本次计算。


In [ ]:
alpha_intervals = alpha_sensitivity[
    ["model", "HAC lags (months)", "HAC standard error (% per month)",
     "95% CI low (% per month)", "95% CI high (% per month)"]
]
display(alpha_intervals)

## 4. 样本方差分解

含截距的 OLS 在本样本内把资产超额收益写成拟合部分与残差之和。下面按样本方差分母 n−1 核对 Var(y) = Var(fitted) + Var(residual)。这是拟合样本的恒等关系，不代表未来残差独立，也不能据此预测下一期。

In [ ]:
variance_rows = []
for name in ("capm", "ff3", "ff5"):
    decomposition = summary["models"][name]["sample_variance_decomposition_ddof1"]
    variance_rows.append({
        "model": name,
        "observed excess variance, ddof=1": decomposition["observed_excess_return"],
        "fitted excess variance, ddof=1": decomposition["fitted_excess_return"],
        "residual variance, ddof=1": decomposition["residual"],
        "fitted + residual": decomposition["sum_fitted_plus_residual"],
        "identity verified": decomposition["identity_holds_within_tolerance"],
    })

display(pd.DataFrame(variance_rows).set_index("model"))

## 5. FF5 暴露与图形

载荷是相对于 FF5 发布因子的估计斜率，单位无量纲；图中区间沿用共享回归核心的 HAC 95% 区间。市场因子是已发布的 Mkt-RF，因此资产超额收益一侧才减 RF。

In [ ]:
display(fits["ff5"].coefficient_table)
print("R²:", fits["ff5"].r_squared)
print("Adjusted R²:", fits["ff5"].adjusted_r_squared)

累加贡献图把每月的因子项、截距项和残差做算术求和，再乘 100 表示百分点。它不是 NAV 或复利财富路径，也不把因子暴露伪装为零美元可投资因子组合。因子收益是发布的组合价差；这里的 beta × 因子收益只是回归归因项。

In [ ]:
from IPython.display import Image, display

figure_info = make_figures(total, frames, fits, output_dir=RUN_DIR)
for key, metadata in figure_info.items():
    image_path = ROOT / metadata["path"]
    print(key, metadata["width_px"], "x", metadata["height_px"], "px")
    display(Image(filename=str(image_path)))

## 6. 结果边界与可复现输出

FF3 与 FF5 在本窗口的同名 Mkt-RF、SMB 数值存在差异，上游原因尚未确定。因此，不把两组回归当作严格嵌套模型进行 F 检验，也不把全部 R² 变化归因于 RMW、CMA。若隔离新增因子的增量，需要另行固定基准因子版本并明确它是自定义组合。

冻结文件是当前发布快照，不是逐月 point-in-time (PIT，时点可得) 历史。样本没有逐股因子重建、交易成本或策略执行测试；结果不构成历史策略表现、净 alpha 或章节完整验收。

In [ ]:
written = write_artifacts(fits, summary, coefficients_path=RUN_DIR / "coefficients.csv", summary_path=RUN_DIR / "summary.json")
print(written)
